## Train a basic MNIST classifier

This workbook will train a simple Feed Forward Neural Network to classify handwritten digits from the famous [MNIST](http://yann.lecun.com/exdb/mnist/) dataset.

### Load MNIST

The MNIST (numbers) dataset is built into PyTorch and can be loaded from using the datasets class.

In [ ]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

In [ ]:
# Define a transform to normalize the data
transform = transforms.Compose([transforms.ToTensor()])

# Download and load the training data
train_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)

# Download and load the test data
test_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)

The data is automatically loaded split into test and training sets.

In [ ]:
print(train_set.data.shape)
print(test_set.data.shape)

The training set contains 60,000 handwritten digits and the test set contains 10,000. Each image is a 28 x 28 pixel greyscale image with 256 intensity levels. We can visualize some examples using the matplotlib package.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

f, ax = plt.subplots(2, 4, figsize=(12,5), squeeze=True)

for r in range(2):
    for c in range(4):
        image_index = r * 4 + c
        ax[r,c].axis("off")
        ax[r,c].imshow(train_set.data[image_index], cmap='gray')

### Prepare the Data

The training and test datasets must be converted from 28 x 28 images with integer greyscale levels to vectors of 784 floating point numbers with values between 0 and 1. To do this the 3D tensor must be flattened to 2D and the values converted to 'float32' type and rescaled.  

In [ ]:
train_x = train_set.data.reshape((60000, 28*28))
train_x = train_x.float() / 255
test_x = test_set.data.reshape((10000, 28 * 28))
test_x = test_x.float() / 255

The vectors of training and test labels contain integer values between 0 and 9. 

In [ ]:
print(train_set.targets)

For the neural network we need the labels to be "one-hot encoded", that is we need to transform each integer value to a vector of length 10 containing binary values. Only one value in the vector is set equal to 1, corresponding to the integer label. We can make this transformation using the <code>to_categorical</code> keras function.

In [ ]:
# Number of classes
num_classes = train_set.targets.max() + 1  # Assuming classes are labeled from 0 to n-1

# Convert to one-hot encoding
train_y = F.one_hot(train_set.targets, num_classes=num_classes).float()
test_y = F.one_hot(test_set.targets, num_classes=num_classes).float()

In [ ]:
print(test_y[0])

### Build Network

Now we can create the Neural Network itself. This will be done by building a Neural Network class that derives from nn.Module. Here we use two layers with 200 neurons in each with the ReLU activation function. Note that the input dimension must be specified in the first layer. The last layer is a linear layer that will be passed through the special *softmax* function used to represent the probability that the input image represents the integers 0 from 9. The softmax is embedded within the cross-entropy loss function in PyTorch. Given this is a probability the sum of the final layer output values will be 1.0. When using the model for inference and evaluation the entry with the highest probability will be used to determine the classification.

In [ ]:
# Define the neural network model as a subclass of nn.Module
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        # Define the first layer with 200 units, the input size is 28*28
        self.fc1 = nn.Linear(28 * 28, 200)
        # Define the second layer with 200 units
        self.fc2 = nn.Linear(200, 200)
        # Define the output layer with 10 units (for 10 classes)
        self.fc3 = nn.Linear(200, 10)

    def forward(self, x):
        # Flatten the input tensor
        x = x.view(-1, 28 * 28)
        # Apply ReLU activation function after first and second layers
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
# Create an instance of the NeuralNetwork
model = NeuralNetwork().to(device)

We need an optimizer and a loss function. In this case *rmsprop* is used as the optimizer along with the *catgorical crossentropy* loss function that is commonly used for multi-class classification problems. Given this is a classification problem it makes sense to track the *accuracy* of classification during training.

In [ ]:
# Define the optimizer (RMSprop in this case)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

# Define the loss function (categorical cross-entropy in this case)
loss_fn = nn.CrossEntropyLoss()

We will use PyTorch DataLoader classes

In [ ]:
train_x = train_x.to(device)
train_y = train_y.to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, 
                                           shuffle=True, drop_last=True)

test_x = test_x.to(device)
test_y = test_y.to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, 
                                          shuffle=False, drop_last=True)

### Train

The model can now be trained using the training examples and labels. The model will be trained through 20 *epochs*, where 1 epoch is an iteration through all the training examples. 

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = torch.argmax(batch_y, dim=1)
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = torch.argmax(batch_y, dim=1)
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

In [ ]:
epochs = 20
history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)

### Evaluate

To evaluate the performance of the model the test set is run through it and the predicted labels compared to the actual labels. The model has not seen the test examples and so this process tests how well the trained model will *generalize* to unseen examples. The loss value and accuracy are displayed, given the accuracy was specified as an additional metric to track during model compilation.

In [ ]:
# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0
correct = 0
with torch.no_grad():  # No gradient computation
    for data, target in test_loader:
        output = model(data)
        y_pred = output.squeeze()
        test_loss += loss_fn(y_pred, target).item()  # Sum up batch loss
        predicted = torch.argmax(y_pred, dim=1)
        targets = torch.argmax(target, dim=1)
        correct += (predicted == targets).sum().item()

test_loss /= len(test_loader.dataset)
test_accuracy = correct / len(test_loader.dataset)

print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {100. * test_accuracy:.2f}%')

### Plot

Finally the training metrics are plotted as a function of epoch. This data is stored in a dictionary in the <code>history</code> object that is returned from the model <code>fit</code> function. 

In [ ]:
acc_values = history_dict['test_acc']
loss_values = history_dict['test_loss']

# Generate the x-axis points
epochs = range(1,len(loss_values) +1)

fig, ax1 = plt.subplots()

# Plot the Accuracy on the left axis
color = 'k'
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Test Accuracy', color=color)
ax1.plot(epochs, acc_values, color=color, linestyle='--')
ax1.tick_params(axis='y', labelcolor=color)

# instantiate a second axes that shares the same x-axis
ax2 = ax1.twinx()  
ax2.set_ylabel('Test Loss', color=color) 
ax2.plot(epochs, loss_values, color=color)
ax2.tick_params(axis='y', labelcolor=color)

# Format and save the figure
fig.tight_layout()
plt.savefig('MNISTBasic.png', dpi=300)